# Tutorial: Execution: Single Run, Batch, and Phases

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Operators running scenario matrices repeatedly.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Execute one variant across phases safely.
- Run batch jobs for all/selected variants.
- Track output folders and rerun reproducibly.


## Outline

1. List variants from selected config
2. Run single variant (reporting/calibration/both)
3. Run batch all variants
4. Run selected variant subset
5. Capture run matrix and output roots


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: List variant names from config


In [ ]:
if load_run_config is None:
    raise RuntimeError("crm_model import missing")
cfg = load_run_config(CONFIG_PATH)
variants = list(cfg.variants.keys())
print("variant count:", len(variants))
for v in variants:
    print(" -", v)


## Step 2: Single run commands


In [ ]:
_ = sh(f"python scripts/run_one.py --config {CONFIG} --variant {EXAMPLE_VARIANT} --phase reporting --save-csv", cwd=REPO)
_ = sh(f"python scripts/run_one.py --config {CONFIG} --variant {EXAMPLE_VARIANT} --phase calibration --save-csv", cwd=REPO)
_ = sh(f"python scripts/run_one.py --config {CONFIG} --variant {EXAMPLE_VARIANT} --phase both --save-csv", cwd=REPO)


## Step 3: Batch run all variants (heavy)


In [ ]:
if RUN_HEAVY:
    _ = sh(f"python scripts/run_batch.py --config {CONFIG} --phase reporting --save-csv --compare", cwd=REPO)
else:
    print("Set RUN_HEAVY=True to execute full batch")


## Step 4: Batch run selected variants


In [ ]:
subset = "baseline,demand_surge,circularity_push"
if RUN_HEAVY:
    _ = sh(f"python scripts/run_batch.py --config {CONFIG} --phase reporting --variants {subset} --save-csv --compare", cwd=REPO)
else:
    print("Selected subset command prepared:", subset)


## Step 5: Manual loop mode


In [ ]:
for v in variants[:3]:
    cmd = f"python scripts/run_one.py --config {CONFIG} --variant {v} --phase reporting --save-csv"
    print(cmd)


## Step 6: Output root inspection


In [ ]:
run_root = REPO / "outputs" / "runs" / CONFIG_STEM
print("Run root:", run_root)
if run_root.exists():
    for p in sorted(run_root.iterdir()):
        if p.is_dir():
            latest = latest_dir(p)
            print(p.name, "latest:", latest.name if latest else "none")


## Pitfalls

- Forgetting `--save-csv` means downstream compare/plot scripts have no inputs.
- Mixing configs (`mvp` vs `r-strategies`) without resetting `CONFIG` causes confusion.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
